In [ ]:
import os
import numpy as np
import tensorflow as tf
from skimage.io import imread
from pycocotools.coco import COCO
import pandas as pd
import h5py


def load_image_and_masks(coco, img_id, img_dir, cat_ids):

    img_info = coco.loadImgs(img_id)[0]
    img_path = os.path.join(img_dir, img_info['file_name'])
    image = imread(img_path) / 255.0


    masks = np.zeros((img_info['height'], img_info['width'], len(cat_ids)))

    for i, cat_id in enumerate(cat_ids):
        ann_ids = coco.getAnnIds(imgIds=img_id, catIds=[cat_id], iscrowd=None)
        anns = coco.loadAnns(ann_ids)
        for ann in anns:
            mask = coco.annToMask(ann)
            masks[..., i] = np.maximum(masks[..., i], mask)

    return image, masks


def load_dataset(coco, img_ids, img_dir, cat_ids, img_size=(128, 128)):


    images = []
    masks = []

    for img_id in img_ids:
        image, mask = load_image_and_masks(coco, img_id, img_dir, cat_ids)
        image_resized = tf.image.resize(image, img_size)

        mask_resized = tf.image.resize(mask, img_size, method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)

        images.append(image_resized)
        masks.append(mask_resized)

    return np.array(images), np.array(masks)


def load_splits_from_csv(csv_file):

    df = pd.read_csv(csv_file)

    train_img_ids = df[df['split_open'] == 'train']['id'].tolist()
    val_img_ids = df[df['split_open'] == 'valid']['id'].tolist()
    test_img_ids = df[df['split_open'] == 'test']['id'].tolist()

    return train_img_ids, val_img_ids, test_img_ids


def save_preprocessed_data(filepath, X_train, y_train, X_val, y_val, X_test, y_test):
    with h5py.File(filepath, 'w') as f:
        f.create_dataset('X_train', data=X_train)
        f.create_dataset('y_train', data=y_train)
        f.create_dataset('X_val', data=X_val)
        f.create_dataset('y_val', data=y_val)
        f.create_dataset('X_test', data=X_test)
        f.create_dataset('y_test', data=y_test)


def load_preprocessed_data(filepath):
    with h5py.File(filepath, 'r') as f:
        X_train = f['X_train'][:]
        y_train = f['y_train'][:]
        X_val = f['X_val'][:]
        y_val = f['y_val'][:]
        X_test = f['X_test'][:]
        y_test = f['y_test'][:]
    return X_train, y_train, X_val, y_val, X_test, y_test


def main():

    data_dir = 'turtles-data/data'
    ann_file = os.path.join(data_dir, 'annotations.json')
    img_dir = data_dir
    csv_file = os.path.join(data_dir, 'metadata_splits.csv')

    print(f"Looking for annotations file at: {ann_file}")

    coco = COCO(ann_file)

    cat_ids = coco.getCatIds()

    train_img_ids, val_img_ids, test_img_ids = load_splits_from_csv(csv_file)

    preprocessed_data_path = "preprocessed_data.h5"

    if os.path.exists(preprocessed_data_path):
        print("Loading preprocessed data from file...")
        X_train, y_train, X_val, y_val, X_test, y_test = load_preprocessed_data(preprocessed_data_path)
    else:
        print("Preprocessing data and saving to file...")

        X_train, y_train = load_dataset(coco, train_img_ids, img_dir, cat_ids)
        X_val, y_val = load_dataset(coco, val_img_ids, img_dir, cat_ids)
        X_test, y_test = load_dataset(coco, test_img_ids, img_dir, cat_ids)


        save_preprocessed_data(preprocessed_data_path, X_train, y_train, X_val, y_val, X_test, y_test)



if __name__ == "__main__":
    main()

Looking for annotations file at: C:/Users/Culac/code/comp9517/Project\annotations.json
loading annotations into memory...
Done (t=6.03s)
creating index...
index created!
Preprocessing data and saving to file...
